<a href="http://landlab.github.io"><img style="float: left; width: 300px;" src="https://landlab.csdms.io/_static/landlab_logo.png"></a>

# 2D Surface Water Flow: HLLC Component — Circular Dam Break

<hr>
<small>For more Landlab tutorials, click here: <a href="https://landlab.readthedocs.io/en/latest/user_guide/tutorials.html">https://landlab.readthedocs.io/en/latest/user_guide/tutorials.html</a></small>
<hr>

## Overview

This notebook demonstrates the `RiverFlowDynamics_HLLC` Landlab component simulating a **circular dam break** — a canonical test for shock-capturing hydraulic solvers. The HLLC scheme is specifically designed for this class of problems: the instantaneous removal of a circular dam creates an outward-propagating **bore** (a moving hydraulic shock), which requires a Riemann-based solver to capture correctly.

### Import the required libraries:

In [ ]:
from IPython.display import clear_output
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import os
from pathlib import Path
from scipy.interpolate import RegularGridInterpolator

from landlab import RasterModelGrid
from landlab.plot.imshow import imshow_grid
from landlab.components import RiverFlowDynamics_HLLC
from river_flow_dynamics_tutorial4_plot import *

## Information about the component

Using the class name as argument for the `help` function returns descriptions of the various methods and parameters.

In [ ]:
help(RiverFlowDynamics_HLLC)

## Example: Circular dam break

This example describes the instantaneous collapse of an idealized circular dam. The initial condition comprises two regions of stationary water separated by a thin-walled cylinder of diameter 22 m located at the center of a 50×50 m² horizontal, frictionless domain. The domain is discretized with 10,201 rectangular elements.

The analytical solution (Stoker 1957) gives the bore front position as $r_\text{bore}(t)$.

### Define simulation parameters

In [ ]:
n = 0.012      # Manning's roughness coefficient, [s/m^(1/3)]
target_time = 0.71  # Total simulated time [s]
nrows = 101    # Number of node rows
ncols = 101    # Number of node cols
dx = 0.5       # Node spacing in x [m]
dy = 0.5       # Node spacing in y [m]

### Create the grid

In [ ]:
grid = RasterModelGrid((nrows, ncols), xy_spacing=(dx, dy))

### Set up topography

Create the elevation field and define the topography to represent a flat surface at the bottom.

In [ ]:
te = grid.add_zeros("topographic__elevation", at="node")

### Set initial water depth

Water depth inside the dam is 11 m; outside is 1 m. A smooth transition zone of 0.5 m avoids a perfectly sharp initial discontinuity (which aids numerical stability while having negligible effect on the bore propagation):

In [ ]:
center_x = 25.0
center_y = 25.0
radius   = 10.5

distance_from_center = np.sqrt((grid.x_of_node - center_x)**2 + (grid.y_of_node - center_y)**2)
inside_circle = distance_from_center < radius

h = grid.add_ones("surface_water__depth", at="node")
h[inside_circle] = 10.0

transition_width = 0.5
transition_zone  = (distance_from_center > radius) & (distance_from_center <= radius + transition_width)
transition_factor = 1 - (distance_from_center[transition_zone] - radius) / transition_width
h[transition_zone] = 1.0 + 9.0 * transition_factor


### Visualize initial conditions

In [ ]:
# -------------------------------------------------
# Reshape node fields
# -------------------------------------------------
nrows, ncols = grid.shape

X = grid.x_of_node.reshape((nrows, ncols))
Y = grid.y_of_node.reshape((nrows, ncols))
H = grid.at_node["surface_water__depth"].reshape((nrows, ncols))

# Flat bed
Zbed = np.zeros_like(H)
WSE = Zbed + H

# Centerline
mid_row = nrows // 2
x_profile = X[mid_row, :]
h_profile = H[mid_row, :]

# 1D coordinates for interpolation
x1d = X[0, :]
y1d = Y[:, 0]

# -------------------------------------------------
# Display-only finer grid for smoother 3D rendering
# -------------------------------------------------
interp_fun = RegularGridInterpolator(
    (y1d, x1d),   # note order: rows -> y, cols -> x
    WSE,
    method="linear",
    bounds_error=False,
    fill_value=None
)

xf = np.linspace(x1d.min(), x1d.max(), 220)
yf = np.linspace(y1d.min(), y1d.max(), 220)
Xf, Yf = np.meshgrid(xf, yf)

pts = np.column_stack([Yf.ravel(), Xf.ravel()])
WSE_fine = interp_fun(pts).reshape(Xf.shape)

# -------------------------------------------------
# Figure
# -------------------------------------------------
fig = plt.figure(figsize=(14, 9), constrained_layout=True)
gs = fig.add_gridspec(2, 2, width_ratios=[1.0, 1.15], height_ratios=[1.0, 0.85])

# -------------------------------------------------
# Panel 1: plan view
# -------------------------------------------------
ax1 = fig.add_subplot(gs[0, 0])

pcm = ax1.pcolormesh(X, Y, H, cmap="Blues", shading="nearest")

dam_circle = Circle(
    (center_x, center_y),
    radius,
    fill=False,
    linestyle="--",
    linewidth=2.0,
    color="black",
    alpha=0.9,
    label="Initial dam radius"
)
ax1.add_patch(dam_circle)

ax1.axhline(center_y, color="white", lw=1.2, ls=":", alpha=0.9)
ax1.axvline(center_x, color="white", lw=1.2, ls=":", alpha=0.9)

ax1.set_aspect("equal", adjustable="box")
ax1.set_xlim(x1d.min(), x1d.max())
ax1.set_ylim(y1d.min(), y1d.max())
ax1.set_title("Plan View: Initial Water Depth", fontsize=13, fontweight="bold")
ax1.set_xlabel("x [m]", fontsize=11)
ax1.set_ylabel("y [m]", fontsize=11)
ax1.legend(loc="upper right", framealpha=1.0)

cbar1 = fig.colorbar(pcm, ax=ax1, fraction=0.046, pad=0.04)
cbar1.set_label("Water Depth [m]")

# -------------------------------------------------
# Panel 2: centerline profile
# -------------------------------------------------
ax2 = fig.add_subplot(gs[1, 0])

ax2.plot(
    x_profile,
    h_profile,
    color="deepskyblue",
    linewidth=2.8,
    label=f"Centerline depth (y = {center_y:.1f} m)",
    zorder=3
)
ax2.fill_between(
    x_profile,
    0.0,
    h_profile,
    color="deepskyblue",
    alpha=0.35,
    zorder=2
)

ax2.axhline(1.0, color="gray", lw=1.2, ls="--", alpha=0.8, label="Outer depth")
ax2.axvline(center_x - radius, color="black", lw=1.2, ls=":", alpha=0.7)
ax2.axvline(center_x + radius, color="black", lw=1.2, ls=":", alpha=0.7)

ax2.set_xlim(x1d.min(), x1d.max())
ax2.set_ylim(0.0, np.max(H) * 1.08)
ax2.set_title("Centerline Profile", fontsize=13, fontweight="bold")
ax2.set_xlabel("Distance [m]", fontsize=11)
ax2.set_ylabel("Water Depth [m]", fontsize=11)
ax2.grid(True, linestyle="--", alpha=0.6, zorder=0)
ax2.legend(loc="upper right", framealpha=1.0)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

# -------------------------------------------------
# Panel 3: smoother 3D display
# -------------------------------------------------
ax3 = fig.add_subplot(gs[:, 1], projection="3d")

surf = ax3.plot_surface(
    Xf, Yf, WSE_fine,
    cmap="Blues",
    edgecolor="none",
    linewidth=0,
    antialiased=True,
    shade=True,
    alpha=0.97
)

# Base contours
ax3.contour(
    Xf, Yf, WSE_fine,
    zdir="z",
    offset=0.0,
    levels=10,
    cmap="Blues",
    linewidths=1.0
)

ax3.set_title("3D View: Initial Water Surface", fontsize=13, fontweight="bold", pad=14)
ax3.set_xlabel("x [m]", labelpad=8)
ax3.set_ylabel("y [m]", labelpad=8)
ax3.set_zlabel("Elevation [m]", labelpad=8)

ax3.set_xlim(x1d.min(), x1d.max())
ax3.set_ylim(y1d.min(), y1d.max())
ax3.set_zlim(0.0, np.max(WSE_fine) * 1.05)

ax3.view_init(elev=28, azim=-135)
ax3.set_box_aspect((1, 1, 0.28))

cbar2 = fig.colorbar(surf, ax=ax3, fraction=0.035, pad=0.04, shrink=0.78)
cbar2.set_label("Water Surface Elevation [m]")

fig.suptitle("Circular Dam-Break Initial Condition", fontsize=16, fontweight="bold")
plt.show()

### Initialize the RiverFlowDynamics_HLLC component

Key differences from `RiverFlowDynamics` for the dam-break case:
- **No inflow/outflow BCs** — this is a closed-domain problem with initial conditions only.
- **Transmissive (open) boundaries by default** — waves can exit the domain without reflection (default behaviour, no `wall_edges` needed here).
- **No `dt`, no `theta`** — CFL-adaptive explicit integration.
- **No pre-created link-velocity or WSE fields**.

In [ ]:
rfd = RiverFlowDynamics_HLLC(grid, mannings_n=n)

### Run the simulation

Each call to `run_one_step()` advances by one CFL-limited adaptive time step. The bore propagates outward; Strang operator splitting ensures isotropic 2D propagation.

In [ ]:
display_frequency = 1
output_dir = "simulation_frames_hllc"
Path(output_dir).mkdir(exist_ok=True)

timestep = 0
while rfd.elapsed_time < target_time:
    print(
        f"Running timestep {timestep+1}  "
        f"(sim t = {rfd.elapsed_time:.3f} s / {target_time:.2f} s, "
        f"dt = {rfd.current_dt:.4f} s)"
    )

    rfd.run_one_step()
    timestep += 1

    if timestep % display_frequency == 0:
        clear_output(wait=True)

        fig, _ = create_combined_plot(
            grid,
            timestep,
            field_name="surface_water__depth",
            center_x=center_x,
            center_y=center_y,
            radius=radius,
            use_smooth_3d=True,
            fine_factor=3,
            figure_title="Circular Dam Break - RiverFlowDynamics_HLLC",
        )

        filename = os.path.join(output_dir, f"frame_{timestep:03d}.png")
        fig.savefig(
            filename,
            dpi=150,
            facecolor="white",
            edgecolor="none"
        )
        plt.close(fig)


### Visualize animation

Let's examine the water depth and water surface elevation evolution in time

In [ ]:
create_gif_animation(
    input_folder="simulation_frames_hllc",
    output_filename="dam_break_hllc.gif",
    total_duration_seconds=3
)
show_gif_in_notebook("dam_break_hllc.gif")

## Interpretation of Results

The circular dam-break results highlight the principal advantage of `RiverFlowDynamics_HLLC` over the semi-implicit `RiverFlowDynamics`:

1. **Bore propagation**: The HLLC solver correctly captures the outward-propagating bore as a sharp discontinuity. The semi-implicit RFD solver smears this bore due to numerical diffusion, underestimating bore speed by ~22%.
2. **Radial symmetry**: Strang operator splitting (alternating x/y sweeps) preserves radial symmetry to second order — the bore front is nearly perfectly circular despite the Cartesian grid.
3. **Positive-depth guarantee**: The hydrostatic reconstruction and positivity limiter ensure no negative depths arise near the bore front, even at coarse resolution.
4. **Velocity from node fields**: x- and y-velocity components (`surface_water__x_velocity`, `surface_water__y_velocity`) at nodes provide the full 2D velocity vector, capturing the radially outward flow pattern.
5. **Adaptive time stepping**: The bore-front region has the largest wave speeds, automatically reducing `dt` to stay within CFL stability.

## Physics of the dam break problem

Key phenomena:

1. **Rarefaction wave**: Propagates inward from the dam wall, reducing the inner water level.
2. **Bore (shock wave)**: Propagates outward, abruptly raising the outer water level.
3. **Mass conservation**: Total water volume is exactly conserved by the finite-volume discretization.
4. **Symmetry breaking**: A Cartesian grid introduces small asymmetries (~grid-scale) visible at sharp fronts; these vanish at finer resolution.

## Conclusion

We demonstrated how `RiverFlowDynamics_HLLC` simulates a circular dam break with high accuracy. The HLLC Riemann solver captures the bore front without artificial smoothing, which is fundamentally impossible for semi-implicit schemes.

-- --
### And that's it!

Nice work completing this tutorial. You now know how to use `RiverFlowDynamics_HLLC` for shock-capturing simulations :)

-- --

### Click here for more <a href="https://landlab.csdms.io/tutorials/">Landlab tutorials</a>